# Instacart Lakehouse Analytics — Business Analysis

## Objective

This notebook uses the curated Gold and Silver tables to create executive KPIs, business insights, customer segments, and dashboard-ready datasets.

The analysis focuses on product demand, department performance, reorder behaviour, customer segments, and ordering patterns by weekday and hour.

## Inputs

- `silver_orders`
- `silver_order_items`
- `gold_customer_summary`
- `gold_product_performance`
- `gold_department_summary`
- `gold_order_behavior`
- `gold_data_quality_report`

## Outputs

- `gold_customer_segmented`
- `dashboard_executive_kpis`
- `dashboard_top_products`
- `dashboard_high_reorder_products`
- `dashboard_department_performance`
- `dashboard_orders_by_day`
- `dashboard_orders_by_hour`
- `dashboard_customer_segments`

## 1. Load Curated Tables

Load the Silver, Gold, and data-quality tables required for business analysis and dashboard preparation.

In [0]:
from pyspark.sql import functions as F

gold_customer = spark.table("workspace.default.gold_customer_summary")
gold_product = spark.table("workspace.default.gold_product_performance")
gold_department = spark.table("workspace.default.gold_department_summary")
gold_order_behavior = spark.table("workspace.default.gold_order_behavior")
quality_report = spark.table("workspace.default.gold_data_quality_report")

## 2. Create Executive KPIs

Calculate high-level metrics that summarize the project dataset:

- total orders
- total customers
- unique products
- total items purchased
- overall reorder rate

In [0]:
# Create a one-row executive summary for dashboard KPI cards
silver_items = spark.table("workspace.default.silver_order_items")
silver_orders = spark.table("workspace.default.silver_orders")

executive_kpis = (
    silver_items
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.countDistinct("user_id").alias("total_customers"),
        F.countDistinct("product_id").alias("unique_products"),
        F.count("*").alias("total_items_purchased"),
        F.avg("reordered").alias("overall_reorder_rate")
    )
)

display(executive_kpis)

total_orders,total_customers,unique_products,total_items_purchased,overall_reorder_rate
109467,54726,36117,1130617,0.6235736770276761


## 3. Calculate Average Basket Size

Aggregate order-item records to the order level and calculate the average number of items per basket.

In [0]:
order_baskets = (
    silver_items
    .groupBy("order_id")
    .agg(F.count("*").alias("basket_size"))
)

average_basket_size = order_baskets.agg(
    F.avg("basket_size").alias("average_basket_size")
)

display(average_basket_size)

average_basket_size
10.328382069482126


## 4. Identify Top Products by Purchase Volume

Rank products by total purchases to highlight the most frequently purchased items in the sample.

In [0]:
top_products = (
    gold_product
    .orderBy(F.desc("total_purchases"))
    .select(
        "product_id",
        "product_name",
        "department",
        "aisle",
        "total_purchases",
        "unique_customers",
        "reorder_rate"
    )
    .limit(20)
)

display(top_products)

product_id,product_name,department,aisle,total_purchases,unique_customers,reorder_rate
24852,Banana,produce,fresh fruits,16495,11821,0.8684449833282814
13176,Bag of Organic Bananas,produce,fresh fruits,13605,9976,0.8620360161705255
21137,Organic Strawberries,produce,fresh fruits,9544,7726,0.8087803855825649
21903,Organic Baby Spinach,produce,packaged vegetables fruits,8369,6757,0.8161070617756004
47209,Organic Hass Avocado,produce,fresh fruits,7227,5720,0.8213643282136432
47766,Organic Avocado,produce,fresh fruits,6061,4750,0.8152120112192708
47626,Large Lemon,produce,fresh fruits,5720,4888,0.7405594405594406
16797,Strawberries,produce,fresh fruits,5048,4238,0.7492076069730587
27966,Organic Raspberries,produce,packaged vegetables fruits,4926,4010,0.785221274868047
26209,Limes,produce,fresh fruits,4898,4226,0.7247856267864434


## 5. Identify Products with Strong Reorder Behaviour

Filter products using a minimum purchase threshold before ranking by reorder rate.

The threshold reduces the risk of low-volume products appearing artificially strong because of very small sample sizes.

In [0]:
# Apply a minimum-volume threshold before ranking products by reorder rate
high_reorder_products = (
    gold_product
    .filter(F.col("total_purchases") >= 100)
    .orderBy(
        F.desc("reorder_rate"),
        F.desc("total_purchases")
    )
    .select(
        "product_name",
        "department",
        "total_purchases",
        "unique_customers",
        "reorder_rate"
    )
    .limit(20)
)

display(high_reorder_products)

product_name,department,total_purchases,unique_customers,reorder_rate
Whole Organic Omega 3 Milk,dairy eggs,320,229,0.90625
Petit Suisse Fruit,dairy eggs,116,91,0.896551724137931
Organic Reduced Fat Omega-3 Milk,dairy eggs,165,120,0.8909090909090909
French Vanilla Soy Creamer,dairy eggs,116,82,0.8879310344827587
Goat Milk,dairy eggs,196,142,0.8724489795918368
"Milk, Organic, Vitamin D",dairy eggs,690,486,0.8710144927536232
Organic Unsweetened Vanilla Almond Milk,dairy eggs,573,424,0.8708551483420593
Organic Original Almond Milk,dairy eggs,355,249,0.8704225352112676
Banana,produce,16495,11821,0.8684449833282814
Organic Reduced Fat Milk,dairy eggs,1185,841,0.8666666666666667


## 6. Analyse Department Performance

Compare departments using purchase volume, customer reach, product breadth, and reorder behaviour.

In [0]:
department_performance = (
    gold_department
    .orderBy(F.desc("total_items_purchased"))
)

display(department_performance)

department_id,department,total_items_purchased,unique_orders,unique_customers,unique_products,reorder_rate,average_cart_position
4,produce,335267,82387,45955,1456,0.686420077132554,8.18610540255975
16,dairy eggs,186533,74501,43674,2914,0.6998493564141466,7.642063334637839
19,snacks,100095,48090,32270,4719,0.6065737549328138,9.349487986412909
7,beverages,93709,50495,33011,3328,0.6792197120874196,7.091464000256112
1,frozen,77709,40873,28395,3224,0.5733184058474565,9.200710342431378
13,pantry,64351,38522,28365,3779,0.38443847026464234,9.749032649065283
3,bakery,40627,30332,21981,1216,0.6629827454648387,8.232357791616412
20,deli,36529,26675,19886,1082,0.6393276574776205,8.890580087054122
15,canned goods,36226,23212,18047,1591,0.5013250151824656,10.139264616573731
9,dry goods pasta,29505,20423,16205,1404,0.49984748347737673,10.506219284866972


## 7. Analyse Orders by Day of Week

Aggregate ordering activity by weekday to identify differences in order volume, basket size, and reorder behaviour.

The source dataset represents weekdays numerically from 0 to 6, where 0 corresponds to Sunday.

In [0]:
orders_by_day = (
    gold_order_behavior
    .groupBy("order_dow")
    .agg(
        F.sum("total_orders").alias("total_orders"),
        F.avg("average_basket_size").alias("average_basket_size"),
        F.avg("average_reorder_rate").alias("average_reorder_rate")
    )
    .orderBy("order_dow")
)

display(orders_by_day)

order_dow,total_orders,average_basket_size,average_reorder_rate
0,20419,11.247349858899545,0.6434558470129731
1,17834,10.329236860617186,0.6283218305463579
2,14380,9.818971092794797,0.619764969466068
3,13643,9.582720116682202,0.6244565610400764
4,13609,9.8772226565188,0.6276901330806804
5,14768,10.526202047878193,0.638198825974485
6,14814,11.04691822270353,0.6248473570814265


## 8. Analyse Orders by Hour of Day

Summarize order activity by hour to identify peak and low-demand periods throughout the day.

In [0]:
orders_by_hour = (
    gold_order_behavior
    .groupBy("order_hour_of_day")
    .agg(
        F.sum("total_orders").alias("total_orders"),
        F.avg("average_basket_size").alias("average_basket_size")
    )
    .orderBy("order_hour_of_day")
)

display(orders_by_hour)

order_hour_of_day,total_orders,average_basket_size
0,722,10.414447563387885
1,392,10.16281471066087
2,247,11.004729658935213
3,181,9.63195050183394
4,165,10.601531965571374
5,319,9.630690249820686
6,1008,10.742540867647156
7,3020,10.69937730885367
8,5824,10.534640047350381
9,8150,10.183679840769873


## 9. Create Rule-Based Customer Segments

Segment sampled customers using order frequency, reorder rate, and average basket size.

The segments are designed for business interpretation rather than predictive modelling:

- Loyal High-Frequency
- Large Basket
- Repeat Buyer
- Low Activity
- Regular Customer

Because customer history is sampled, these segments should be interpreted as project-sample behavioural groups.

In [0]:
# Assign interpretable customer segments using rule-based business thresholds
gold_customer_segmented = (
    gold_customer
    .withColumn(
        "customer_segment",
        F.when(
            (F.col("sampled_order_count") >= 4) &
            (F.col("customer_reorder_rate") >= 0.6),
            "Loyal High-Frequency"
        )
        .when(
            F.col("average_basket_size") >= 15,
            "Large Basket"
        )
        .when(
            F.col("customer_reorder_rate") >= 0.5,
            "Repeat Buyer"
        )
        .when(
            F.col("sampled_order_count") <= 1,
            "Low Activity"
        )
        .otherwise("Regular Customer")
    )
)

## 10. Summarize Customer Segments

Aggregate the segmented customer table to compare segment size, average basket size, and average reorder rate.

In [0]:
customer_segment_summary = (
    gold_customer_segmented
    .groupBy("customer_segment")
    .agg(
        F.countDistinct("user_id").alias("customers"),
        F.avg("average_basket_size").alias("average_basket_size"),
        F.avg("customer_reorder_rate").alias("average_reorder_rate")
    )
    .orderBy(F.desc("customers"))
)

display(customer_segment_summary)

customer_segment,customers,average_basket_size,average_reorder_rate
Repeat Buyer,26860,7.388696548239518,0.7595210855971173
Large Basket,11224,20.71752197671659,0.6026787133906463
Regular Customer,9568,7.785697364229981,0.2949965645702809
Low Activity,4639,7.116835524897607,0.16269414700649923
Loyal High-Frequency,2435,10.292893484566989,0.7969230517194102


In [0]:
gold_customer_segmented.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.gold_customer_segmented")

## 11. Create Dashboard-Ready Tables

Persist compact, purpose-built tables for the Databricks AI/BI dashboard.

These tables reduce dashboard complexity by pre-aggregating the required metrics.

In [0]:
# Save pre-aggregated datasets for efficient dashboard visualizations
analysis_tables = {
    "dashboard_executive_kpis": executive_kpis,
    "dashboard_top_products": top_products,
    "dashboard_high_reorder_products": high_reorder_products,
    "dashboard_department_performance": department_performance,
    "dashboard_orders_by_day": orders_by_day,
    "dashboard_orders_by_hour": orders_by_hour,
    "dashboard_customer_segments": customer_segment_summary
}

for table_name, df in analysis_tables.items():
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"workspace.default.{table_name}")
    )

    print(f"Created: workspace.default.{table_name}")

Created: workspace.default.dashboard_executive_kpis
Created: workspace.default.dashboard_top_products
Created: workspace.default.dashboard_high_reorder_products
Created: workspace.default.dashboard_department_performance
Created: workspace.default.dashboard_orders_by_day
Created: workspace.default.dashboard_orders_by_hour
Created: workspace.default.dashboard_customer_segments


## Business Analysis Summary

The curated Instacart data was transformed into executive KPIs, product and department rankings, weekday and hourly order patterns, and rule-based customer segments.

### Dashboard-ready tables created

- `dashboard_executive_kpis`
- `dashboard_top_products`
- `dashboard_high_reorder_products`
- `dashboard_department_performance`
- `dashboard_orders_by_day`
- `dashboard_orders_by_hour`
- `dashboard_customer_segments`

The resulting datasets support the published Databricks AI/BI dashboard and provide the analytical foundation for the Power BI version of the project.